# Linear Representation Hypothesis

**Survey Reference:** Section 6.1 - The Linear Representation Hypothesis

Features correspond to directions $\phi \in \mathbb{R}^d$ in activation space. The projection $\phi^\top h_l(x)$ approximately measures the intensity of that feature for input $x$, and features combine additively: $h_l(x) \approx \sum_i \alpha_i \phi_i$. This implies features can be detected via linear probes and added or removed via vector arithmetic on the $h_l(x)$.

In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sklearn.linear_model import LogisticRegression
import numpy as np

# Setup Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"Model loaded on {device}")

Model loaded on cuda


## 1. Extracting Activations

We define a helper function to extract residual stream activations at a specific layer. For each input text, we retrieve the hidden state of the **last token** at the specified layer.

In [2]:
def get_activations(text, layer_idx):
    """Extracts the residual stream activations at a specific layer."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        # activations shape: (batch, seq_len, hidden_dim)
        # We take the activation of the LAST token
        hidden_states = outputs.hidden_states[layer_idx][0, -1, :]
    return hidden_states

## 2. Computing a Feature Direction (Difference-in-Means)

According to the Linear Representation Hypothesis, concepts correspond to **directions** $\phi \in \mathbb{R}^d$ in activation space. We extract a "sentiment direction" using the **Difference-in-Means** method:

$$\phi = \frac{1}{|P|}\sum_{x \in P} h_l(x) - \frac{1}{|N|}\sum_{x \in N} h_l(x)$$

where $P$ is a set of positive (happy) examples and $N$ is a set of negative (sad) examples.

In [3]:
# Using an earlier layer often works better for steering
layer_to_probe = 5

# Define contrastive prompt sets
happy_prompts = ["I am so happy", "Life is wonderful", "Feeling great today", 
                 "This is amazing", "I love this", "What a joy"]
sad_prompts = ["I am so sad", "Life is terrible", "Feeling down today",
               "This is awful", "I hate this", "What a disaster"]

# Collect activations for each set
happy_acts = torch.stack([get_activations(p, layer_to_probe) for p in happy_prompts])
sad_acts = torch.stack([get_activations(p, layer_to_probe) for p in sad_prompts])

# Feature Direction phi = mean(Positive) - mean(Negative)
sentiment_direction = (happy_acts.mean(dim=0) - sad_acts.mean(dim=0))
phi = sentiment_direction / torch.norm(sentiment_direction)  # Normalize


## 3. Detection: Linear Probing

If the Linear Representation Hypothesis holds, we can **detect** the presence and intensity of a concept by computing the projection $\phi^\top h_l(x)$. A higher positive value indicates more "happiness" in the activation, while a negative value indicates more "sadness".

In [4]:
# Test the projection: φᵀ · h
test_prompts = ["I feel amazing", "I feel terrible", "The weather is nice"]

print("Sentiment Intensity Detection")
print("=" * 50)
for test_prompt in test_prompts:
    h_test = get_activations(test_prompt, layer_to_probe)
    intensity = torch.dot(phi, h_test).item()
    sentiment = "positive" if intensity > 0 else "negative"
    print(f"'{test_prompt}'")
    print(f"  → Intensity: {intensity:+.4f} ({sentiment})")
    print()

Sentiment Intensity Detection
'I feel amazing'
  → Intensity: +5.8029 (positive)

'I feel terrible'
  → Intensity: -15.2764 (negative)

'The weather is nice'
  → Intensity: +1.2135 (positive)



## 4. Intervention: Vector Arithmetic (Activation Steering)

The Linear Representation Hypothesis also implies we can **intervene** on the model's behavior by adding or subtracting concept directions. We modify the hidden state during inference:

$$h'_l(x) = h_l(x) + \alpha \cdot \phi$$

where $\alpha$ controls the steering strength:
- $\alpha > 0$: steer toward "happiness"
- $\alpha < 0$: steer toward "sadness"
- $\alpha = 0$: no modification (baseline)

In [5]:
def steer_model(prompt, alpha=5.0, layer=None):
    """Generate text with activation steering applied."""
    if layer is None:
        layer = layer_to_probe
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Define a hook to add our direction to the hidden state during inference
    def hook_fn(module, input, output):
        # output is a tuple (hidden_states, ...)
        # We add alpha * phi to ALL token positions
        modified_hidden = output[0].clone()
        modified_hidden[:, :, :] += alpha * phi.to(modified_hidden.device)
        return (modified_hidden,) + output[1:]

    # Register hook to the specific layer
    handle = model.transformer.h[layer].register_forward_hook(hook_fn)
    
    # Generate text with the steered activation
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=15, 
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=1.0,
            top_p=0.9
        )
    result = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    handle.remove()  # Clean up the hook
    return result

### Steering Experiment

We apply the steering function to a neutral prompt with varying $\alpha$ values to observe how the model's output changes when we push the activations toward "happiness" or "sadness".

In [6]:
neutral_prompt = "I think that today is"

print("STEERING EXPERIMENT")
print("=" * 60)
print(f"Prompt: '{neutral_prompt}'")
print("=" * 60)

# Test with different steering strengths
for alpha in [0, 25, 50, -25, -50]:
    output = steer_model(neutral_prompt, alpha=alpha)
    if alpha == 0:
        direction = "neutral (baseline)"
    elif alpha > 0:
        direction = f"→ happy (α={alpha})"
    else:
        direction = f"→ sad (α={alpha})"
    print(f"\n{direction}:")
    print(f"  {output}")
    print("-" * 60)

STEERING EXPERIMENT
Prompt: 'I think that today is'

neutral (baseline):
  I think that today is the right time to take on the job," she said.

The
------------------------------------------------------------

→ happy (α=25):
  I think that today is the beginning of the era of the Internet," he added.

This
------------------------------------------------------------

→ happy (α=50):
  I think that today is a great occasion for you to have fun working with us, our friends at
------------------------------------------------------------

→ sad (α=-25):
  I think that today is a national embarrassment, which I think has made this economy and the people there
------------------------------------------------------------

→ sad (α=-50):
  I think that today is pretty serious," Mr. Ticapacoglou, a former head
------------------------------------------------------------


## Limitations and Considerations

The Linear Representation Hypothesis serves as a useful approximation rather than a universal law. Several important limitations should be considered:

- **Nonlinear feature interactions**: Some concepts may require complex, nonlinear combinations of features that cannot be captured through simple linear projections
- **Algorithmic representations**: Certain types of information processing may require nonlinear transformations that go beyond additive feature combinations  
- **Distributed representations**: Features may be distributed across multiple interacting subspaces rather than concentrated in single directions

Additionally, features in neural networks may not necessarily exist as explicit, linear representations. Instead, they might emerge as implicit patterns from the underlying computational dynamics of the network circuits.

Understanding these limitations helps us apply the Linear Representation Hypothesis more effectively as an interpretability tool while remaining aware of its boundaries.

→ **02_superposition_hypothesis.ipynb**: What happens when there are more concepts than dimensions?